### FASE MODEL

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import textwrap
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import accuracy_score, roc_curve, auc, confusion_matrix

# --- CARGA Y CONFIGURACION DE DATOS ---
ARCHIVO = "../Scrub/datos_Scrub.csv"
OBJETIVO = "likert_prob_steam_binario"
ALPHA = 0.05
COLORS = {"ink": "#1F2C38", "acc": "#C0392B", "grey": "#A5B3BF", "grid": "#D8DEE4"}
plt.rcParams["font.family"] = "DejaVu Sans"

df = pd.read_csv(ARCHIVO)
print(f"Muestra Total: {len(df)} | Prefiere STEAM: {df[OBJETIVO].sum()} | No prefiere: {len(df) - df[OBJETIVO].sum()}\n")

# --- ASOCIACIÓN DE VARIABLES  ---
def cramers_v(x, y):
    tabla = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(tabla)[0]
    return np.sqrt((chi2 / tabla.sum().sum()) / min(tabla.shape[0] - 1, tabla.shape[1] - 1))

# Diccionarios de variables

vars_ordinales ={"edu_padres_num":"Edu Padres","ingreso_familiar_num": "Ingreso Familiar","opinion_amigos_num":"Opinion de Amigos",    "nivel_informacion_steam_num":"Nivel Informacion STEAM",    "año_escolar_num": "Año Escolar","likert_optimismo": "Optimismo","likert_apoyo_familia": "Apoyo Familiar", "likert_estabilidad_eco": "Estabilidad Eco.","likert_percep_tecnologia":"Percepcion Tec.","likert_apoyo_maestros": "Apoyo de Maestros"
    
}
vars_nominales = {
    "impacto_esperado_num": "Impacto Esperado", "obstaculos_num": "Obstaculos",
    "sectores_interes_num": "Sectores de interes", "area_tecnol_importante_num": "Area Tecnologica"
}

vars_dicotomicas = {
    "edad": "EDAD", "genero_num": "GENERO", "area_residencia_num": "AREA de RESIDENCIA",
    "q9_herramientas": "Herramientas", "q9_construccion_fisica": "Bricolaje", "q9_mecanica": "Mecanica", 
    "q9_seres_vivos": "Seres Vivos", "q9_ingenieria": "Ingenieria", "q9_fenomenos_f": "Fenomenos fisicos",
    "q9_ajedrez": "Ajedrez", "q9_ciencia_ficcion": "Ciencia ficcion", "q9_logica_deductiva": "Novelas", 
    "q9_artes_plasticas": "Temas artisticos", "q9_arte_visual": "Arte Visual", 
    "q9_actividades_artisticas": "Actvidades Artisticas", "q10_analisis_datos": "Analisis de Datos", 
    "q10_descubrimientos_cient": "Descubrimientos", "q10_explorar_tecnologia": "Exp Tecnologia", 
    "q10_problemas_software": "Problemas Software", "q10_diseño_construccion": "Diseño y Construccion", 
    "q10_problemas_creativos": "Problemas Creativos", "q10_artes_visuales": "Artes visuales", 
    "q10_retroalim_arte": "Retro Arte", "q10_estadistica": "Estadistica", 
    "q10_problemas_matematicos": "Problem Matematicos", "q11_matematicas": "Asig Matematicas", 
    "q11_biologia": "Asig Biologia", "q11_fisica": "Asig Fisica", "q11_quimica": "Asig Quimica", 
    "q11_tecnologia": "Asig Tecnologia", "q11_arte": "Asig Arte", "q12_investigador": "Investigador Cient",
    "q12_desarrollador": "Dev Software", "q12_ingeniero_civil": "Ingeniero Civil", 
    "q12_artista_digital": "Artista digital", "q12_matematico": "Matematico", "q12_otro": "Otra"
}

resultados = []

# 1. Cálculos de asociación (Nominales -> V de Cramér)
for col, nombre in vars_nominales.items():
    if col in df.columns:
        resultados.append({"Variable": nombre, "Métrica": "V Cramér", 
                           "Valor": cramers_v(df[col], df[OBJETIVO]), 
                           "Valor p": stats.chi2_contingency(pd.crosstab(df[col], df[OBJETIVO]))[1]})

# 2. Cálculos de asociación (Dicotómicas -> Point-Biserial / Phi)
for col, nombre in vars_dicotomicas.items():
    if col in df.columns:
        phi, p = stats.pointbiserialr(df[col], df[OBJETIVO])
        resultados.append({"Variable": nombre, "Métrica": "phi", "Valor": phi, "Valor p": p})

# 3. Cálculos de asociación (Ordinales -> Correlación de Spearman)
for col, nombre in vars_ordinales.items():
    if col in df.columns:
        rho, p = stats.spearmanr(df[col], df[OBJETIVO])
        resultados.append({"Variable": nombre, "Métrica": "Spearman (rho)", "Valor": rho, "Valor p": p})

# Exportar variables
df_res = pd.DataFrame(resultados)
with pd.ExcelWriter("Tabla_Resultados.xlsx") as writer:
    df_res.to_excel(writer, sheet_name="Todas_las_variables", index=False)
    df_res[df_res["Valor p"] < ALPHA].to_excel(writer, sheet_name="Solo_Significativas", index=False)

print("[1/4] Asociaciones bivariadas exportadas a Tabla_Resultados.xlsx")

# --- 2. CORRECCIÓN DE HOLM-BONFERRONI ---
pruebas = {
    "Optimismo": stats.spearmanr(df['likert_optimismo'], df[OBJETIVO])[1],
    "Apoyo familia": stats.spearmanr(df['likert_apoyo_familia'], df[OBJETIVO])[1],
    "Estabilidad eco": stats.spearmanr(df['likert_estabilidad_eco'], df[OBJETIVO])[1],
    "Percep. tecnologia": stats.spearmanr(df['likert_percep_tecnologia'], df[OBJETIVO])[1],
    "Apoyo maestros": stats.spearmanr(df['likert_apoyo_maestros'], df[OBJETIVO])[1],
    "Ingreso familiar": stats.spearmanr(df['ingreso_familiar_num'], df[OBJETIVO])[1],
    "Impacto esperado": stats.chi2_contingency(pd.crosstab(df['impacto_esperado_num'], df[OBJETIVO]))[1],
    "AREA de RESIDENCIA": stats.pointbiserialr(df['area_residencia_num'], df[OBJETIVO])[1],
    "Gusto por Analisis de Datos": stats.pointbiserialr(df['q10_analisis_datos'], df[OBJETIVO])[1],
    "Asig Matematicas": stats.pointbiserialr(df['q11_matematicas'], df[OBJETIVO])[1],
    "Ingeniero civil": stats.pointbiserialr(df['q12_ingeniero_civil'], df[OBJETIVO])[1],
}

nombres, pvals = list(pruebas.keys()), list(pruebas.values())
rechaza, p_ajust, _, _ = multipletests(pvals, alpha=ALPHA, method="holm")

df_holm = pd.DataFrame({
    "Variable": nombres, "Valor p Original": pvals, "Valor p Ajustado": p_ajust,
    "Estado": ["SOBREVIVE" if r else "No sobrevive" for r in rechaza]
}).sort_values("Valor p Original")

df_holm.to_excel("Resultados_Holm.xlsx", index=False)
print("[2/4] Corrección de Holm exportada a Resultados_Holm.xlsx")

# --- 3. REGRESIÓN LOGÍSTICA ---
PREDICTORES = ["likert_optimismo", "likert_apoyo_familia", "likert_estabilidad_eco", 
            "likert_percep_tecnologia", "likert_apoyo_maestros", "ingreso_familiar_num", 
            "q11_matematicas", "q10_analisis_datos", "q12_ingeniero_civil", 
            "area_residencia_num", "impacto_esperado_num"]

X = sm.add_constant(df[PREDICTORES].astype(float))
y = df[OBJETIVO]

modelo = sm.Logit(y, X).fit(disp=0)
ic = np.exp(modelo.conf_int())
pred_probs = modelo.predict(X)
pred_bin = (pred_probs > 0.5).astype(int)

df_logit = pd.DataFrame({
    "Predictor": X.columns,
    "OR": np.exp(modelo.params),
    "IC 95% Inf": ic[0],
    "IC 95% Sup": ic[1],
    "Valor p": modelo.pvalues,
    "Significativo": ["Sí" if p < ALPHA else "No" for p in modelo.pvalues]
}).round(4)

df_logit.to_excel("Resultados_Regresion.xlsx", index=False)
print(f"[3/4] Modelo exportado (R2: {modelo.prsquared:.3f}, Exactitud: {accuracy_score(y, pred_bin):.1%})")

# Validaciones del Modelo (VIF y Hosmer-Lemeshow)
print("\n--- Validaciones del Modelo ---")
print("VIF (Multicolinealidad):")
for i, col in enumerate(X.columns[1:], 1):
    print(f" - {col}: {variance_inflation_factor(X.values, i):.2f}")

hl_df = pd.DataFrame({"y": y, "p": pred_probs}).assign(g=lambda x: pd.qcut(x["p"], 10, labels=False, duplicates="drop"))
obs, exp, n_g = hl_df.groupby("g")["y"].sum(), hl_df.groupby("g")["p"].sum(), hl_df.groupby("g")["y"].count()
hl_stat = (((obs - exp) ** 2) / (exp * (1 - exp / n_g))).sum()
print(f"Hosmer-Lemeshow (Bondad de ajuste): p = {1 - stats.chi2.cdf(hl_stat, hl_df['g'].nunique() - 2):.3f}\n")

# --- 4. VISUALIZACIONES ---
print("[4/4] Generando Gráficos...")

# 4.1 Mapa de Calor (Solo sobrevivientes Holm)
mapa_nombres = {
    "Optimismo": "likert_optimismo", "Apoyo familia": "likert_apoyo_familia", "Estabilidad eco": "likert_estabilidad_eco",
    "Percep. tecnologia": "likert_percep_tecnologia", "Apoyo maestros": "likert_apoyo_maestros",
    "Ingreso familiar": "ingreso_familiar_num", "Impacto esperado": "impacto_esperado_num",
    "AREA de RESIDENCIA": "area_residencia_num", "Gusto por Analisis de Datos": "q10_analisis_datos",
    "Asig Matematicas": "q11_matematicas", "Ingeniero civil": "q12_ingeniero_civil"
}

sobreviven = df_holm[df_holm["Estado"] == "SOBREVIVE"]["Variable"].tolist()
cols_hm = [OBJETIVO] + [mapa_nombres[v] for v in sobreviven if v in mapa_nombres]

if len(cols_hm) > 1:
    plt.figure(figsize=(8, 6))
    
    # 1. Unimos "Prob. STEAM" con las variables que sobrevivieron
    etiquetas_crudas = ["Prob. STEAM"] + sobreviven
    
    # 2. ALTERNATIVA: Reemplazamos los espacios normales por "Enters" (\n)
    etiquetas_limpias = [texto.replace(" ", "\n") for texto in etiquetas_crudas]
    
    # 3. Dibujamos el gráfico pasándole nuestras etiquetas limpias
    sns.heatmap(df[cols_hm].corr(method='spearman'), annot=True, fmt=".2f", cmap="RdBu_r", 
                vmin=-1, vmax=1, square=True, 
                xticklabels=etiquetas_limpias, yticklabels=etiquetas_limpias)
    
    plt.title("Correlación (Solo significativas post-Holm)\n")
    
    # 4. Forzamos a que el texto quede totalmente horizontal
    plt.xticks(rotation=0, fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    
    plt.tight_layout()
    plt.savefig("Grafico_1_Mapa_Calor.png", dpi=300)
    plt.close()

# 4.2 Forest Plot
df_plot = df_logit[df_logit["Predictor"] != "const"].sort_values("OR")
plt.figure(figsize=(8, 6))

# 1. Invertimos tu diccionario automáticamente para que traduzca de "código" a "Nombre Legible"
mapa_inverso = {valor: llave for llave, valor in mapa_nombres.items()}

# 2. Creamos una lista con los nombres traducidos (si no encuentra uno, deja el original)
etiquetas_y = [mapa_inverso.get(nombre, nombre) for nombre in df_plot["Predictor"]]

plt.errorbar(df_plot["OR"], range(len(df_plot)), 
            xerr=[df_plot["OR"]-df_plot["IC 95% Inf"], df_plot["IC 95% Sup"]-df_plot["OR"]], 
            fmt='o', color=COLORS["acc"], ecolor=COLORS["grey"], capsize=4)

plt.axvline(1, color=COLORS["ink"], linestyle='--')

# 3. Le pasamos nuestras nuevas etiquetas traducidas al eje Y
plt.yticks(range(len(df_plot)), etiquetas_y)

plt.title('Forest Plot: Odds Ratios')
plt.grid(axis='x', linestyle=':')
plt.tight_layout()
plt.savefig("Grafico_2_Forest.png", dpi=300); plt.close()
# 4.3 Curva ROC
fpr, tpr, _ = roc_curve(y, pred_probs)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color=COLORS["acc"], lw=2, label=f'AUC = {auc(fpr, tpr):.3f}')
plt.plot([0, 1], [0, 1], color=COLORS["grey"], linestyle='--')
plt.title('Curva ROC'), plt.legend(loc="lower right"), plt.grid(linestyle=':')
plt.tight_layout()
plt.savefig("Grafico_3_ROC.png", dpi=300); plt.close()

# 4.4 Matriz de Confusión
plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y, pred_bin), annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title('Matriz de Confusión'), plt.xlabel('Predicción'), plt.ylabel('Real')
plt.tight_layout()
plt.savefig("Grafico_4_Confusion.png", dpi=300); plt.close()

print("¡Proceso finalizado! Excel y Gráficos guardados exitosamente.")